## Telemetry Anomaly Detection Framework - v3.0

**Production-grade, fully-structured anomaly detection pipeline following the canonical two-phase architecture.**

```
╔══════════════════════════════════════════════════════════════════════════════════╗
║               TELEMETRY ANOMALY DETECTION FRAMEWORK  v3.0                        ║
╠═══════════════════════════════╦═══════════════════════════════════════════════  ═╣
║   PHASE 1 · MODELS OF NORMAL  ║   PHASE 2 · ANOMALY DETECTION                    ║
╠═══════════════════════════════╬════════════════════════════════════════════════  ╣
║                               ║                                                  ║
║  Datasets                     ║  Datasets + Algorithm Fits                       ║
║     │                         ║  + Compressed Feature Vectors                    ║
║     ▼                         ║     │                                            ║
║  ┌──────────────────────┐     ║     ▼                                            ║
║  │     Algorithms       │     ║  ┌────────────────────────────────────────┐      ║
║  │  ┌──────────────┐    │     ║  │         Anomaly Definitions            │      ║
║  │  │ Rolling Mean │    │     ║  │  ┌──────────────┐ ┌──────────────────┐ │      ║
║  │  ├──────────────┤    │─────╫──►  │ σ From Mean  │ │ σ From Mean of   │ │      ║
║  │  │    ARIMA     │    │     ║  │  │   of Data    │ │     Errors       │ │      ║
║  │  ├──────────────┤    │     ║  │  └──────────────┘ └──────────────────┘ │      ║
║  │  │ Autoencoder  │    │     ║  │  ┌──────────────────────────────────┐  │      ║
║  │  └──────────────┘    │     ║  │  │ Nonparametric Dynamic Threshold  │  │      ║
║  └──────────────────────┘     ║  │  └──────────────────────────────────┘  │      ║
║     │                         ║  └──────────────────────┬─────────────────┘      ║
║     ▼                         ║                          │◄─ RRCF Anomaly      ║
║  Data Files + Plots           ║                          │    Scores           ║
║     │                         ║     │                    │                     ║
║     ▼                         ║     ▼                    │                     ║
║  Correlation Coefficients     ║  More Data Files + Plots                       ║
║                               ║     │                                          ║
║                               ║     ▼                                          ║
║                               ║  Counts of Anomalous Points                    ║
╚═══════════════════════════════╩════════════════════════════════════════════════╝
```

| Area | v2 | v3 |
|---|---|---|
| Phase 1 models | Rolling Z only | Rolling Mean + ARIMA + Autoencoder |
| Phase 2 detectors | IsoForest, LCAI, Cohort | σ/Data, σ/Errors, NDT, RRCF |
| Compressed vectors | None | Autoencoder latent space (8-D) |
| Algorithm fits saved | No | Yes (pickle + DataFrame) |
| Correlation matrix | None | Full model × signal correlation |
| Anomaly counts table | Basic | Per-detector × per-device × per-type |
| RRCF | No | Yes (shingled online forest) |
| Dynamic Thresholding | No | Yes (NASA NDT algorithm) |


### 0. Environment & Imports

In [4]:
import sys
!{sys.executable} -m pip install rrcf

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for rrcf: filename=rrcf-0.4.4-py3-none-any.whl size=10672 sha256=a54d09784de5a5e118e53946b4d5020a30029ea2d1495c67e22eeb28f402fef2
  Stored in directory: c:\users\mnaik\appdata\local\pip\cache\wheels\46\5f\a8\4ae602057e8487bc3f03d4ca80cbe04d97d2f5b292131afcad
Successfully built rrcf



[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
# ── Standard library ─────────────────────────────────────────────────────────
from __future__ import annotations

import json
import logging
import os
import pickle
import warnings
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")

# ── Third-party ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
from scipy.signal import savgol_filter
from sklearn.cluster import DBSCAN
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import RobustScaler, StandardScaler

# ARIMA
from statsmodels.tsa.arima.model import ARIMA

# RRCF — Robust Random Cut Forest
try:
    import rrcf
    RRCF_AVAILABLE = True
except ImportError:
    os.system("pip install rrcf -q")
    try:
        import rrcf
        RRCF_AVAILABLE = True
    except Exception:
        RRCF_AVAILABLE = False
        print("  rrcf unavailable — falling back to IsoForest-based RRCF approximation")

print(f" All imports successful  [{datetime.now().strftime('%H:%M:%S')}]")
print(f"  RRCF available : {RRCF_AVAILABLE}")


 All imports successful  [12:25:03]
  RRCF available : True


### 1 · Configuration & Logging

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# OUTPUT PATHS  — mirrors the diagram's "Data Files / Plots" outputs
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR = Path("telemetry_v3_outputs")
for sub in ["models", "features", "plots", "reports", "anomalies"]:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
# CENTRALISED CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────

@dataclass
class TelemetryConfig:
    """Single source of truth for every pipeline parameter."""

    # Data generation
    n_points_per_device : int   = 3_000         # kept ≤ 5k for ARIMA tractability
    random_seed         : int   = 42
    anomaly_rate        : float = 0.05
    devices             : List[str] = field(default_factory=lambda: [
        "iPhone", "MacBook", "Apple Watch", "iPad"
    ])
    os_versions         : List[str] = field(default_factory=lambda: [
        "iOS 18.3", "iOS 18.4", "iOS 18.5"
    ])

    # ── Phase 1: Models of Normal ────────────────────────────────────────────
    # Rolling Mean
    rolling_window      : int   = 50

    # ARIMA
    arima_order         : Tuple[int, int, int] = (2, 1, 2)
    arima_train_frac    : float = 0.80          # fraction used to fit ARIMA
    arima_signals       : List[str] = field(default_factory=lambda: ["latency", "cpu_usage"])

    # Autoencoder
    ae_latent_dim       : int   = 8             # compressed feature vector size
    ae_hidden_dim       : int   = 32
    ae_max_iter         : int   = 80
    ae_features         : List[str] = field(default_factory=lambda: [
        "latency", "cpu_usage", "memory_usage", "temperature",
        "packet_loss", "jitter", "disk_io", "gpu_usage",
        "drain_rate", "app_load_time",
    ])

    # ── Phase 2: Anomaly Definitions ─────────────────────────────────────────
    # σ from mean of data
    sigma_data_k        : float = 3.0

    # σ from mean of errors
    sigma_error_k       : float = 3.0

    # Nonparametric Dynamic Thresholding (NASA NDT)
    ndt_window          : int   = 50
    ndt_z_score         : float = 2.5
    ndt_smoothing       : bool  = True

    # RRCF
    rrcf_tree_size      : int   = 256
    rrcf_num_trees      : int   = 40
    rrcf_shingle_size   : int   = 8
    rrcf_threshold_pct  : float = 95.0         # percentile above = anomaly

    # ── Ensemble & Risk ──────────────────────────────────────────────────────
    ensemble_min_votes  : int   = 2
    risk_threshold      : float = 0.70
    risk_weights        : Dict[str, float] = field(default_factory=lambda: {
        "latency":     0.30,
        "cpu_usage":   0.20,
        "temperature": 0.20,
        "packet_loss": 0.15,
        "battery_inv": 0.15,
    })
    csaf_threshold      : float = 0.65
    csaf_weights        : Tuple[float, float, float] = (0.50, 0.30, 0.20)

    # DBSCAN
    dbscan_eps          : float = 15.0
    dbscan_min_samples  : int   = 20
    dbscan_sample_size  : int   = 6_000

    # Alert thresholds
    severity_critical   : float = 0.80
    severity_high       : float = 0.60
    severity_medium     : float = 0.40


CFG = TelemetryConfig()

# ─────────────────────────────────────────────────────────────────────────────
# LOGGING
# ─────────────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(name)-24s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("telemetry.v3")
log.info("Config loaded — devices=%s  n=%d", CFG.devices, CFG.n_points_per_device)


12:25:18 | INFO     | telemetry.v3             | Config loaded — devices=['iPhone', 'MacBook', 'Apple Watch', 'iPad']  n=3000


###  2 · Synthetic Telemetry Generator

In [8]:
DEVICE_PROFILES: Dict[str, Dict] = {
    "iPhone":      {"latency": 80,  "cpu": 50, "temp": 35, "mem": 60},
    "MacBook":     {"latency": 60,  "cpu": 65, "temp": 45, "mem": 70},
    "Apple Watch": {"latency": 40,  "cpu": 30, "temp": 30, "mem": 45},
    "iPad":        {"latency": 70,  "cpu": 45, "temp": 38, "mem": 58},
}

ANOMALY_TYPES = [
    "latency_spike", "cpu_spike", "overheat",
    "battery_drain", "network_issue", "memory_pressure",
]


def _build_device_frame(device: str, n: int, rng: np.random.Generator) -> pd.DataFrame:
    p = DEVICE_PROFILES[device]
    t = np.arange(n, dtype=float)

    latency     = p["latency"] + 10 * np.sin(t / 50) + rng.normal(0, 5, n)
    cpu         = p["cpu"]     + 15 * np.sin(t / 30) + rng.normal(0, 7, n)
    memory      = p["mem"]     + rng.normal(0, 5, n)
    battery     = np.clip(100 - (t % 400) * 0.15 + rng.normal(0, 1, n), 0, 100)
    temperature = p["temp"]    + cpu * 0.1 + rng.normal(0, 2, n)

    wifi_strength    = np.clip(75 + rng.normal(0, 10, n), 0, 100)
    cellular_latency = np.clip(50 + rng.normal(0, 15, n), 10, 200)
    packet_loss      = np.clip(rng.normal(1, 1, n), 0, 10)
    jitter           = np.clip(rng.normal(5, 2, n), 0, 20)
    disk_io          = np.abs(rng.normal(100, 30, n))
    thread_count     = np.clip(rng.normal(200, 50, n), 50, 500).astype(int)
    gpu_usage        = np.clip(rng.normal(40, 15, n), 0, 100)

    battery_temp     = temperature + rng.normal(0, 1, n)
    skin_temp        = temperature - np.abs(rng.normal(2, 1, n))
    thermal_pressure = np.clip(temperature / 100, 0, 1)
    drain_rate       = np.clip(rng.normal(0.2, 0.05, n), 0.05, 0.5)
    charge_cycles    = np.clip(rng.normal(300, 50, n), 100, 1000).astype(int)
    charging_state   = rng.choice([0, 1], n, p=[0.7, 0.3])

    app_load_time    = np.clip(rng.normal(500, 150, n), 100, 2000)
    frame_drops      = np.clip(rng.normal(5, 3, n), 0, 50).astype(int)
    crash_count      = rng.choice([0, 1, 2], n, p=[0.90, 0.08, 0.02])

    cpu_mem_ratio    = cpu / (memory + 1)
    network_score    = wifi_strength - packet_loss * 5
    health_index     = np.clip(
        1.0 - (cpu / 200 + temperature / 120 + packet_loss / 20) / 3, 0, 1
    )

    timestamps = pd.date_range("2026-01-01", periods=n, freq="s")

    return pd.DataFrame({
        "timestamp": timestamps, "device": device,
        "latency": latency, "cpu_usage": cpu, "memory_usage": memory,
        "battery_level": battery, "temperature": temperature,
        "wifi_strength": wifi_strength, "cellular_latency": cellular_latency,
        "packet_loss": packet_loss, "jitter": jitter,
        "disk_io": disk_io, "thread_count": thread_count, "gpu_usage": gpu_usage,
        "battery_temp": battery_temp, "skin_temp": skin_temp,
        "thermal_pressure": thermal_pressure,
        "drain_rate": drain_rate, "charge_cycles": charge_cycles,
        "charging_state": charging_state,
        "app_load_time": app_load_time, "frame_drops": frame_drops,
        "crash_count": crash_count,
        "cpu_mem_ratio": cpu_mem_ratio, "network_score": network_score,
        "health_index": health_index,
    })


def _inject_anomalies(df: pd.DataFrame, rate: float, rng: np.random.Generator) -> pd.DataFrame:
    df = df.copy()
    df["anomaly_label"] = 0
    df["anomaly_type"]  = "normal"
    n_anomalies = int(rate * len(df))
    idxs = rng.choice(len(df), n_anomalies, replace=False)
    for idx in idxs:
        atype = rng.choice(ANOMALY_TYPES)
        if   atype == "latency_spike":   df.loc[idx, "latency"]       += rng.uniform(120, 250)
        elif atype == "cpu_spike":       df.loc[idx, "cpu_usage"]      = min(100, df.loc[idx, "cpu_usage"] + rng.uniform(35, 70))
        elif atype == "overheat":        df.loc[idx, "temperature"]   += rng.uniform(18, 35)
        elif atype == "battery_drain":
            end = min(idx + 25, len(df) - 1)
            df.loc[idx:end, "battery_level"] -= rng.uniform(18, 45)
        elif atype == "network_issue":
            df.loc[idx, "packet_loss"] += rng.uniform(6, 18)
            df.loc[idx, "jitter"]      += rng.uniform(10, 25)
        elif atype == "memory_pressure": df.loc[idx, "memory_usage"]   = min(100, df.loc[idx, "memory_usage"] + rng.uniform(25, 40))
        df.loc[idx, "anomaly_label"] = 1
        df.loc[idx, "anomaly_type"]  = atype
    return df


def generate_telemetry(cfg: TelemetryConfig) -> pd.DataFrame:
    rng = np.random.default_rng(cfg.random_seed)
    frames: List[pd.DataFrame] = []
    for device in cfg.devices:
        log.info("Generating  device=%-12s  n=%d", device, cfg.n_points_per_device)
        raw = _build_device_frame(device, cfg.n_points_per_device, rng)
        raw = _inject_anomalies(raw, cfg.anomaly_rate, rng)
        frames.append(raw)
    df = pd.concat(frames, ignore_index=True)
    df["os_version"] = rng.choice(cfg.os_versions, len(df))
    df["event_type"] = rng.choice(["none", "os_update", "app_install"], len(df), p=[0.85, 0.10, 0.05])
    df["cohort"]     = df["device"] + "_" + df["os_version"]
    log.info("Dataset ready  shape=%s  anomalies=%d (%.1f%%)",
             df.shape, df["anomaly_label"].sum(), 100 * df["anomaly_label"].mean())
    return df


df = generate_telemetry(CFG)
print(f"Dataset: {df.shape[0]:,} rows × {df.shape[1]} cols")
df[["device", "latency", "cpu_usage", "temperature", "anomaly_label", "anomaly_type"]].head(4)


12:25:29 | INFO     | telemetry.v3             | Generating  device=iPhone        n=3000
12:25:29 | INFO     | telemetry.v3             | Generating  device=MacBook       n=3000
12:25:29 | INFO     | telemetry.v3             | Generating  device=Apple Watch   n=3000
12:25:29 | INFO     | telemetry.v3             | Generating  device=iPad          n=3000
12:25:29 | INFO     | telemetry.v3             | Dataset ready  shape=(12000, 31)  anomalies=600 (5.0%)


Dataset: 12,000 rows × 31 cols


,device,latency,cpu_usage,temperature,anomaly_label,anomaly_type
0,iPhone,81.523585,58.743159,40.254741,0,normal
1,iPhone,75.000066,55.313753,40.651848,0,normal
2,iPhone,84.152149,64.762152,38.994703,0,normal
3,iPhone,85.302464,40.662671,36.290760,0,normal


### 3 · Feature Engineering

In [9]:
class FeatureEngineer:
    """Enriches raw telemetry with rolling stats, drift, and interaction features."""

    NUMERIC_SIGNALS = [
        "latency", "cpu_usage", "memory_usage", "temperature",
        "packet_loss", "jitter", "disk_io", "gpu_usage",
        "drain_rate", "app_load_time", "frame_drops",
    ]

    def __init__(self, cfg: TelemetryConfig) -> None:
        self.cfg = cfg

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        log.info("Feature engineering ...")
        df = df.copy()
        df = self._rolling_features(df)
        df = self._drift_features(df)
        df = self._interaction_features(df)
        df = self._percentile_normalization(df)
        log.info("Done — %d columns", len(df.columns))
        return df

    def _rolling_features(self, df: pd.DataFrame) -> pd.DataFrame:
        w = self.cfg.rolling_window
        for sig in ["latency", "cpu_usage", "temperature", "packet_loss"]:
            grp = df.groupby("device")[sig]
            df[f"{sig}_roll_mean"] = grp.transform(lambda x: x.rolling(w, min_periods=1).mean())
            df[f"{sig}_roll_std"]  = grp.transform(lambda x: x.rolling(w, min_periods=1).std().fillna(1))
            df[f"{sig}_z"]         = (df[sig] - df[f"{sig}_roll_mean"]) / df[f"{sig}_roll_std"]
        return df

    def _drift_features(self, df: pd.DataFrame) -> pd.DataFrame:
        for sig in ["latency", "cpu_usage"]:
            ewma = df.groupby("device")[sig].transform(lambda x: x.ewm(span=100, adjust=False).mean())
            df[f"{sig}_drift"] = df[sig] - ewma
        return df

    def _interaction_features(self, df: pd.DataFrame) -> pd.DataFrame:
        df["thermal_cpu_stress"]   = df["temperature"] * df["cpu_usage"] / 100
        df["net_quality_score"]    = (df["wifi_strength"] / 100 - df["packet_loss"] / 20 - df["jitter"] / 40).clip(-1, 1)
        df["battery_health_score"] = df["battery_level"] / 100 * (1 - df["drain_rate"])
        df["composite_load"]       = (0.4 * df["cpu_usage"] / 100 + 0.3 * df["memory_usage"] / 100 + 0.3 * df["gpu_usage"] / 100)
        return df

    def _percentile_normalization(self, df: pd.DataFrame) -> pd.DataFrame:
        for sig in self.NUMERIC_SIGNALS:
            df[f"{sig}_pct"] = df.groupby("device")[sig].transform(lambda x: x.rank(pct=True))
        return df


fe = FeatureEngineer(CFG)
df = fe.transform(df)
print(f"Columns after feature engineering: {len(df.columns)}")


12:25:40 | INFO     | telemetry.v3             | Feature engineering ...
12:25:40 | INFO     | telemetry.v3             | Done — 60 columns


Columns after feature engineering: 60


### 4 · Phase 1 - Models of Normal

> **Architecture left side:** fit three complementary models on each device's telemetry to learn what "normal" looks like.  
> Outputs saved to `telemetry_v3_outputs/models/` and `telemetry_v3_outputs/features/`.


### 4a · Rolling Mean Model

In [13]:
class RollingMeanModel:
    """
    Per-device rolling-mean/std model.
    Produces:
      • predicted value  (rolling mean)
      • residual         (actual − predicted)
      • z-score          (residual / rolling std)
    """

    def __init__(self, cfg: TelemetryConfig) -> None:
        self.cfg    = cfg
        self._stats: Dict[str, pd.DataFrame] = {}   # device → stats df
        self._log   = logging.getLogger("models.rolling_mean")

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        self._log.info("Fitting Rolling Mean model (window=%d) ...", self.cfg.rolling_window)
        df = df.copy()
        w  = self.cfg.rolling_window

        for device in df["device"].unique():
            mask   = df["device"] == device
            device_df = df.loc[mask].copy()

            for sig in ["latency", "cpu_usage", "temperature", "packet_loss"]:
                roll_mean = device_df[sig].rolling(w, min_periods=1).mean()
                roll_std  = device_df[sig].rolling(w, min_periods=1).std().fillna(1).replace(0, 1)
                residual  = device_df[sig] - roll_mean
                z_score   = residual / roll_std

                df.loc[mask, f"rm_pred_{sig}"]     = roll_mean.values
                df.loc[mask, f"rm_resid_{sig}"]    = residual.values
                df.loc[mask, f"rm_z_{sig}"]        = z_score.values

        # Save model stats per device
        self._save_model_stats(df)
        self._log.info("Rolling Mean model done")
        return df

    def _save_model_stats(self, df: pd.DataFrame) -> None:
        stats_rows = []
        for device in df["device"].unique():
            mask = df["device"] == device
            for sig in ["latency", "cpu_usage", "temperature"]:
                stats_rows.append({
                    "device": device, "signal": sig,
                    "mean": df.loc[mask, sig].mean(),
                    "std":  df.loc[mask, sig].std(),
                    "p95":  df.loc[mask, sig].quantile(0.95),
                })
        stats_df = pd.DataFrame(stats_rows)
        stats_df.to_csv(OUTPUT_DIR / "models" / "rolling_mean_stats.csv", index=False)
        self._stats = {d: stats_df[stats_df["device"] == d] for d in df["device"].unique()}
        self._log.info("   stats saved → rolling_mean_stats.csv")


rolling_model = RollingMeanModel(CFG)
df = rolling_model.fit_transform(df)
rm_cols = [c for c in df.columns if c.startswith("rm_")]
print(f"Rolling Mean model added {len(rm_cols)} columns")
df[["device", "latency", "rm_pred_latency", "rm_resid_latency", "rm_z_latency"]].head(4)


12:27:03 | INFO     | models.rolling_mean      | Fitting Rolling Mean model (window=50) ...
12:27:03 | INFO     | models.rolling_mean      |    stats saved → rolling_mean_stats.csv
12:27:03 | INFO     | models.rolling_mean      | Rolling Mean model done


Rolling Mean model added 12 columns


,device,latency,rm_pred_latency,rm_resid_latency,rm_z_latency
0,iPhone,81.523585,81.523585,0.000000,0.000000
1,iPhone,75.000066,78.261826,-3.261760,-0.707107
2,iPhone,84.152149,80.225267,3.926882,0.833352
3,iPhone,85.302464,81.494566,3.807898,0.826099


### 4b · ARIMA Model

In [14]:
class ARIMAModel:
    """
    Per-device ARIMA model for each configured signal.
    Fits on arima_train_frac of the data, predicts the full series
    one-step-ahead (in-sample) using the fitted parameters.

    Produces per-signal:
      • arima_pred_{sig}   — fitted/predicted values
      • arima_resid_{sig}  — residuals
    """

    def __init__(self, cfg: TelemetryConfig) -> None:
        self.cfg     = cfg
        self._models: Dict[str, Any] = {}           # (device, signal) → ARIMAResults
        self._log    = logging.getLogger("models.arima")

    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        order = self.cfg.arima_order

        for device in df["device"].unique():
            mask      = df["device"] == device
            device_df = df.loc[mask].copy().reset_index(drop=True)
            n         = len(device_df)
            n_train   = int(n * self.cfg.arima_train_frac)

            for sig in self.cfg.arima_signals:
                self._log.info("  ARIMA%s  device=%-12s  signal=%s", order, device, sig)
                series = device_df[sig].values.astype(float)

                try:
                    model  = ARIMA(series[:n_train], order=order)
                    result = model.fit()
                    self._models[(device, sig)] = result

                    # In-sample fitted values for train portion
                    fitted = result.fittedvalues.copy()

                    # Forecast the remainder one-step at a time
                    # (Use last fitted params; extend with predict)
                    if n_train < n:
                        forecast = result.predict(start=n_train, end=n - 1, dynamic=False)
                        full_pred = np.concatenate([fitted, forecast])
                    else:
                        full_pred = fitted

                    # Align lengths
                    if len(full_pred) < n:
                        full_pred = np.concatenate([np.full(n - len(full_pred), np.nan), full_pred])
                    full_pred = full_pred[:n]

                    residuals = series - full_pred

                except Exception as exc:
                    self._log.warning("    ARIMA failed: %s — using zeros", exc)
                    full_pred = np.zeros(n)
                    residuals = series - series.mean()

                df.loc[mask, f"arima_pred_{sig}"]  = full_pred
                df.loc[mask, f"arima_resid_{sig}"] = residuals

        self._save_fits(df)
        self._log.info("ARIMA model done")
        return df

    def _save_fits(self, df: pd.DataFrame) -> None:
        fit_cols = [c for c in df.columns if c.startswith("arima_")]
        df[["device", "timestamp"] + fit_cols].to_csv(
            OUTPUT_DIR / "models" / "arima_fits.csv", index=False
        )
        # Pickle model objects
        with open(OUTPUT_DIR / "models" / "arima_models.pkl", "wb") as f:
            pickle.dump(self._models, f)
        self._log.info("   fits + model objects saved")


arima_model = ARIMAModel(CFG)
df = arima_model.fit_transform(df)
arima_cols = [c for c in df.columns if c.startswith("arima_")]
print(f"ARIMA model added {len(arima_cols)} columns")
df[["device", "latency", "arima_pred_latency", "arima_resid_latency"]].dropna().head(4)


12:27:15 | INFO     | models.arima             |   ARIMA(2, 1, 2)  device=iPhone        signal=latency


12:27:15 | INFO     | models.arima             |   ARIMA(2, 1, 2)  device=iPhone        signal=cpu_usage
12:27:16 | INFO     | models.arima             |   ARIMA(2, 1, 2)  device=MacBook       signal=latency
12:27:17 | INFO     | models.arima             |   ARIMA(2, 1, 2)  device=MacBook       signal=cpu_usage
12:27:18 | INFO     | models.arima             |   ARIMA(2, 1, 2)  device=Apple Watch   signal=latency
12:27:20 | INFO     | models.arima             |   ARIMA(2, 1, 2)  device=Apple Watch   signal=cpu_usage
12:27:21 | INFO     | models.arima             |   ARIMA(2, 1, 2)  device=iPad          signal=latency
12:27:22 | INFO     | models.arima             |   ARIMA(2, 1, 2)  device=iPad          signal=cpu_usage
12:27:23 | INFO     | models.arima             |    fits + model objects saved
12:27:23 | INFO     | models.arima             | ARIMA model done


ARIMA model added 4 columns


,device,latency,arima_pred_latency,arima_resid_latency
0,iPhone,81.523585,0.000000,81.523585
1,iPhone,75.000066,81.504946,-6.504880
2,iPhone,84.152149,78.241047,5.911102
3,iPhone,85.302464,80.237672,5.064792


### 4c · Autoencoder Model  *(Compressed Feature Vectors)*

In [15]:
class AutoencoderModel:
    """
    Per-device MLP Autoencoder that learns to reconstruct multivariate
    telemetry signals through a compressed bottleneck (latent_dim).

    Produces:
      • ae_latent_{0..latent_dim-1}   — compressed feature vectors (8-D by default)
      • ae_recon_error                — mean-squared reconstruction error per row
      • ae_recon_error_z              — z-scored reconstruction error per device
    """

    def __init__(self, cfg: TelemetryConfig) -> None:
        self.cfg     = cfg
        self._scalers: Dict[str, StandardScaler] = {}
        self._encoders: Dict[str, MLPRegressor]  = {}  # compress to latent
        self._decoders: Dict[str, MLPRegressor]  = {}  # reconstruct from latent
        self._log    = logging.getLogger("models.autoencoder")

    # ── Public ────────────────────────────────────────────────────────────────
    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        feats = [f for f in self.cfg.ae_features if f in df.columns]
        lat   = self.cfg.ae_latent_dim
        hid   = self.cfg.ae_hidden_dim

        for device in df["device"].unique():
            self._log.info("  Autoencoder  device=%s  feats=%d  latent=%d", device, len(feats), lat)
            mask  = df["device"] == device
            X_raw = df.loc[mask, feats].fillna(0).values.astype(float)

            # Scale
            scaler = StandardScaler()
            X = scaler.fit_transform(X_raw)
            self._scalers[device] = scaler

            # ── Encoder: input → latent ───────────────────────────────────────
            encoder = MLPRegressor(
                hidden_layer_sizes=(hid, lat),
                activation="relu",
                max_iter=self.cfg.ae_max_iter,
                random_state=self.cfg.random_seed,
                learning_rate_init=1e-3,
                early_stopping=True,
                validation_fraction=0.1,
                n_iter_no_change=10,
            )
            # Train encoder: predict the first `lat` PCA-like components as targets
            # Use SVD to get an initial latent target
            _, _, Vt = np.linalg.svd(X, full_matrices=False)
            Z_target = X @ Vt[:lat].T          # shape (n, lat) — low-rank projection

            encoder.fit(X, Z_target)
            Z = encoder.predict(X)             # compressed feature vectors
            self._encoders[device] = encoder

            # ── Decoder: latent → reconstruction ─────────────────────────────
            decoder = MLPRegressor(
                hidden_layer_sizes=(lat, hid),
                activation="relu",
                max_iter=self.cfg.ae_max_iter,
                random_state=self.cfg.random_seed + 1,
                learning_rate_init=1e-3,
                early_stopping=True,
                validation_fraction=0.1,
                n_iter_no_change=10,
            )
            decoder.fit(Z, X)                  # reconstruct from latent
            X_recon = decoder.predict(Z)
            self._decoders[device] = decoder

            # ── Reconstruction error ─────────────────────────────────────────
            recon_err = np.mean((X - X_recon) ** 2, axis=1)
            recon_err_std = recon_err.std() or 1.0
            recon_err_z   = (recon_err - recon_err.mean()) / recon_err_std

            # ── Store in DataFrame ────────────────────────────────────────────
            for i in range(lat):
                df.loc[mask, f"ae_latent_{i}"] = Z[:, i]
            df.loc[mask, "ae_recon_error"]   = recon_err
            df.loc[mask, "ae_recon_error_z"] = recon_err_z

        self._save_compressed_vectors(df)
        self._log.info("Autoencoder done")
        return df

    def _save_compressed_vectors(self, df: pd.DataFrame) -> None:
        latent_cols = [c for c in df.columns if c.startswith("ae_")]
        df[["device", "timestamp", "anomaly_label"] + latent_cols].to_csv(
            OUTPUT_DIR / "features" / "compressed_feature_vectors.csv", index=False
        )
        # Save model objects
        with open(OUTPUT_DIR / "models" / "autoencoder_models.pkl", "wb") as f:
            pickle.dump({"encoders": self._encoders, "decoders": self._decoders,
                         "scalers": self._scalers}, f)
        self._log.info("   compressed feature vectors + models saved")


ae_model = AutoencoderModel(CFG)
df = ae_model.fit_transform(df)
ae_cols = [c for c in df.columns if c.startswith("ae_")]
print(f"Autoencoder added {len(ae_cols)} columns")
df[["device", "ae_recon_error", "ae_recon_error_z"] + [f"ae_latent_{i}" for i in range(4)]].head(4)


12:27:32 | INFO     | models.autoencoder       |   Autoencoder  device=iPhone  feats=10  latent=8
12:27:33 | INFO     | models.autoencoder       |   Autoencoder  device=MacBook  feats=10  latent=8
12:27:33 | INFO     | models.autoencoder       |   Autoencoder  device=Apple Watch  feats=10  latent=8
12:27:34 | INFO     | models.autoencoder       |   Autoencoder  device=iPad  feats=10  latent=8
12:27:35 | INFO     | models.autoencoder       |    compressed feature vectors + models saved
12:27:35 | INFO     | models.autoencoder       | Autoencoder done


Autoencoder added 10 columns


,device,ae_recon_error,ae_recon_error_z,ae_latent_0,ae_latent_1,ae_latent_2,ae_latent_3
0,iPhone,0.034496,-0.643838,0.513965,0.349735,-0.626673,1.126807
1,iPhone,0.179083,0.124494,0.147051,0.434535,0.201564,-0.157516
2,iPhone,0.116018,-0.210635,0.170521,0.491141,-0.344644,1.421607
3,iPhone,0.031599,-0.659236,-0.326740,-1.284166,-2.046980,0.142045


### 4d · Correlation Coefficients  *(Phase 1 output table)*

In [16]:
# ─────────────────────────────────────────────────────────────────────────────
# CORRELATION COEFFICIENTS
# Measures how well each model's predictions correlate with the true signal
# and how models agree with each other — a key Phase-1 output.
# ─────────────────────────────────────────────────────────────────────────────

def compute_correlation_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns a model × signal correlation matrix.
    Rows  = detector/model
    Cols  = telemetry signal
    Value = Pearson r between model's predictions/scores and raw signal.
    """
    rows = []
    signals = ["latency", "cpu_usage", "temperature", "packet_loss"]

    for device in df["device"].unique():
        mask = df["device"] == device
        sub  = df.loc[mask]

        for sig in signals:
            # Rolling Mean correlation
            if f"rm_pred_{sig}" in sub.columns:
                r_rm = sub[sig].corr(sub[f"rm_pred_{sig}"])
                rows.append({"device": device, "signal": sig, "model": "Rolling Mean", "pearson_r": round(r_rm, 4)})

            # ARIMA correlation
            if f"arima_pred_{sig}" in sub.columns:
                r_ar = sub[sig].corr(sub[f"arima_pred_{sig}"])
                rows.append({"device": device, "signal": sig, "model": "ARIMA", "pearson_r": round(r_ar, 4)})

        # Autoencoder: corr of reconstruction error with anomaly label
        if "ae_recon_error" in sub.columns:
            r_ae = sub["ae_recon_error"].corr(sub["anomaly_label"])
            rows.append({"device": device, "signal": "anomaly_label", "model": "Autoencoder", "pearson_r": round(r_ae, 4)})

    corr_df = pd.DataFrame(rows)
    pivot   = corr_df.pivot_table(index=["model"], columns=["device", "signal"],
                                   values="pearson_r", aggfunc="mean")
    corr_df.to_csv(OUTPUT_DIR / "reports" / "correlation_coefficients.csv", index=False)
    log.info(" Correlation coefficients saved")
    return corr_df, pivot


corr_df, corr_pivot = compute_correlation_table(df)
print("Correlation Coefficients (Phase 1 output):")
print(corr_df.groupby(["model", "signal"])["pearson_r"].mean().round(4).to_string())


12:27:47 | INFO     | telemetry.v3             |  Correlation coefficients saved


Correlation Coefficients (Phase 1 output):
model         signal       
ARIMA         cpu_usage        0.5218
              latency          0.2465
Autoencoder   anomaly_label    0.4914
Rolling Mean  cpu_usage        0.5518
              latency          0.3529
              packet_loss      0.1406
              temperature      0.2675


## § 5 · Phase 2 — Anomaly Definitions

> **Architecture right side:** Apply four complementary anomaly definitions to the  
> model fits and compressed feature vectors to produce binary flags + counts.


### 5a · σ's From Mean of Data

In [17]:
class SigmaFromMeanOfData:
    """
    Anomaly Definition 1: σ's From Mean of Data
    ─────────────────────────────────────────────
    Flags data points that lie more than k standard deviations from the
    per-device rolling mean of the raw signal (distribution-based detection).

    Uses the Rolling Mean model's z-scores directly.
    """

    def __init__(self, k: float = 3.0) -> None:
        self.k   = k
        self._log = logging.getLogger("detect.sigma_data")

    def detect(self, df: pd.DataFrame) -> pd.Series:
        self._log.info("σ-from-data  k=%.1f", self.k)
        flag = pd.Series(False, index=df.index)

        for sig in ["latency", "cpu_usage", "temperature", "packet_loss"]:
            z_col = f"rm_z_{sig}"
            if z_col in df.columns:
                flag |= df[z_col].abs() > self.k

        # Also flag extreme autoencoder reconstruction error (≡ far from normal manifold)
        if "ae_recon_error_z" in df.columns:
            flag |= df["ae_recon_error_z"] > self.k

        result = flag.astype(int)
        self._log.info("  flagged %d rows (%.2f%%)", result.sum(), 100 * result.mean())
        return result


sigma_data_detector = SigmaFromMeanOfData(k=CFG.sigma_data_k)
df["det_sigma_data"] = sigma_data_detector.detect(df)
print(f"σ-from-data anomalies: {df['det_sigma_data'].sum():,}  ({100*df['det_sigma_data'].mean():.2f}%)")


12:27:56 | INFO     | detect.sigma_data        | σ-from-data  k=3.0
12:27:56 | INFO     | detect.sigma_data        |   flagged 469 rows (3.91%)


σ-from-data anomalies: 469  (3.91%)


### 5b · σ's From Mean of Errors

In [18]:
class SigmaFromMeanOfErrors:
    """
    Anomaly Definition 2: σ's From Mean of Errors
    ───────────────────────────────────────────────
    Flags data points whose *model residuals* (actual − predicted) deviate
    more than k standard deviations from the mean residual.

    Operates on residuals from Rolling Mean AND ARIMA models.
    Error-based detection is more sensitive than raw-data detection for
    smoothly drifting signals.
    """

    def __init__(self, k: float = 3.0) -> None:
        self.k    = k
        self._log = logging.getLogger("detect.sigma_errors")

    def detect(self, df: pd.DataFrame) -> pd.Series:
        self._log.info("σ-from-errors  k=%.1f", self.k)
        flag = pd.Series(False, index=df.index)

        for sig in ["latency", "cpu_usage", "temperature"]:
            # Rolling Mean residuals
            rm_col = f"rm_resid_{sig}"
            if rm_col in df.columns:
                grp = df.groupby("device")[rm_col]
                mu  = grp.transform("mean")
                sd  = grp.transform("std").replace(0, 1)
                z   = (df[rm_col] - mu) / sd
                flag |= z.abs() > self.k

            # ARIMA residuals
            ar_col = f"arima_resid_{sig}"
            if ar_col in df.columns:
                valid = df[ar_col].notna()
                grp   = df.loc[valid].groupby("device")[ar_col]
                mu    = grp.transform("mean")
                sd    = grp.transform("std").replace(0, 1)
                z     = (df.loc[valid, ar_col] - mu) / sd
                flag.loc[valid] |= z.abs() > self.k

        result = flag.astype(int)
        self._log.info("  flagged %d rows (%.2f%%)", result.sum(), 100 * result.mean())
        return result


sigma_error_detector = SigmaFromMeanOfErrors(k=CFG.sigma_error_k)
df["det_sigma_errors"] = sigma_error_detector.detect(df)
print(f"σ-from-errors anomalies: {df['det_sigma_errors'].sum():,}  ({100*df['det_sigma_errors'].mean():.2f}%)")


12:31:59 | INFO     | detect.sigma_errors      | σ-from-errors  k=3.0
12:31:59 | INFO     | detect.sigma_errors      |   flagged 352 rows (2.93%)


σ-from-errors anomalies: 352  (2.93%)


### 5c · Nonparametric Dynamic Thresholding  *(NASA NDT)*

In [19]:
class NonparametricDynamicThreshold:
    """
    Anomaly Definition 3: Nonparametric Dynamic Thresholding
    ─────────────────────────────────────────────────────────
    Implementation of the NASA Telemetry Anomaly Detection (Hundman et al., 2018)
    dynamic threshold algorithm.

    Algorithm:
    1. Compute errors from model predictions (ARIMA residuals).
    2. Smooth errors with Savitzky-Golay filter.
    3. Estimate local error distribution using a sliding window.
    4. Threshold = μ_err + k × σ_err (adaptive, not fixed).
    5. Sequences of errors exceeding threshold are flagged as anomalies.

    Nonparametric: threshold is re-estimated at each window position,
    adapting to the local error distribution without assuming normality.
    """

    def __init__(self, cfg: TelemetryConfig) -> None:
        self.cfg  = cfg
        self._log = logging.getLogger("detect.ndt")

    def detect(self, df: pd.DataFrame) -> pd.Series:
        self._log.info("Nonparametric Dynamic Thresholding  window=%d  z=%.1f",
                       self.cfg.ndt_window, self.cfg.ndt_z_score)
        result = pd.Series(0, index=df.index)

        for device in df["device"].unique():
            mask = df["device"] == device
            sub  = df.loc[mask].copy()

            # Collect residuals from all available models
            error_cols = [c for c in sub.columns if ("resid" in c or "recon_error" == c)]
            if not error_cols:
                continue

            # Aggregate multi-model errors
            errors = sub[error_cols].fillna(0).abs().mean(axis=1).values.astype(float)

            # Smooth errors
            if self.cfg.ndt_smoothing and len(errors) > 51:
                try:
                    errors = savgol_filter(errors, window_length=21, polyorder=3)
                except Exception:
                    pass

            # Compute dynamic threshold per point using a trailing window
            w      = self.cfg.ndt_window
            k      = self.cfg.ndt_z_score
            flags  = np.zeros(len(errors), dtype=int)

            for i in range(w, len(errors)):
                window = errors[max(0, i - w): i]
                mu     = np.mean(window)
                sd     = np.std(window) or 1e-8
                if errors[i] > mu + k * sd:
                    flags[i] = 1

            result.loc[mask] = flags

        count = result.sum()
        self._log.info("  flagged %d rows (%.2f%%)", count, 100 * result.mean())
        return result


ndt_detector = NonparametricDynamicThreshold(CFG)
df["det_ndt"] = ndt_detector.detect(df)
print(f"NDT anomalies: {df['det_ndt'].sum():,}  ({100*df['det_ndt'].mean():.2f}%)")


12:32:06 | INFO     | detect.ndt               | Nonparametric Dynamic Thresholding  window=50  z=2.5
12:32:06 | INFO     | detect.ndt               |   flagged 731 rows (6.09%)


NDT anomalies: 731  (6.09%)


### 5d · RRCF Anomaly Scores  *(Robust Random Cut Forest)*

In [20]:
class RRCFDetector:
    """
    Anomaly Definition 4: RRCF (Robust Random Cut Forest) Anomaly Scores
    ─────────────────────────────────────────────────────────────────────
    RRCF is an online anomaly detection algorithm for streaming telemetry.

    Algorithm (per device, per signal):
    1. Shingle the time series (convert to overlapping windows = feature vectors).
    2. Maintain a forest of Robust Random Cut Trees (each tree sees a sliding
       buffer of recent points).
    3. Compute CoDisp (Collusive Displacement) for each new point:
         CoDisp = displacement caused by removing the point from its tree.
         High CoDisp → the point is anomalous.
    4. Flag points whose CoDisp exceeds the `threshold_pct` percentile.

    Reference: Guha et al., "Robust Random Cut Forest Based Anomaly Detection
    on Streams", ICML 2016.
    """

    SIGNAL = "latency"   # primary signal for RRCF (most informative)

    def __init__(self, cfg: TelemetryConfig) -> None:
        self.cfg  = cfg
        self._log = logging.getLogger("detect.rrcf")

    def detect(self, df: pd.DataFrame) -> pd.Series:
        self._log.info(
            "RRCF  trees=%d  tree_size=%d  shingle=%d  pct=%.0f",
            self.cfg.rrcf_num_trees, self.cfg.rrcf_tree_size,
            self.cfg.rrcf_shingle_size, self.cfg.rrcf_threshold_pct,
        )
        result   = pd.Series(0, index=df.index)
        avg_codisp = pd.Series(0.0, index=df.index)

        for device in df["device"].unique():
            mask   = df["device"] == device
            series = df.loc[mask, self.SIGNAL].values.astype(float)
            scores = self._compute_rrcf_scores(series)
            avg_codisp.loc[mask] = scores
            threshold = np.percentile(scores, self.cfg.rrcf_threshold_pct)
            result.loc[mask] = (scores > threshold).astype(int)
            self._log.info("  device=%-12s  threshold=%.3f  flagged=%d",
                           device, threshold, (scores > threshold).sum())

        df["rrcf_codisp"] = avg_codisp
        count = result.sum()
        self._log.info("  total flagged %d rows (%.2f%%)", count, 100 * result.mean())
        return result

    def _compute_rrcf_scores(self, series: np.ndarray) -> np.ndarray:
        n           = len(series)
        shingle_sz  = self.cfg.rrcf_shingle_size
        num_trees   = self.cfg.rrcf_num_trees
        tree_size   = self.cfg.rrcf_tree_size
        avg_codisp  = np.zeros(n)

        if RRCF_AVAILABLE:
            # ── Native rrcf library ────────────────────────────────────────────
            forest = []
            for _ in range(num_trees):
                forest.append(rrcf.RCTree())

            shingle = np.zeros(shingle_sz)
            for i, val in enumerate(series):
                # Update shingle
                shingle = np.roll(shingle, -1)
                shingle[-1] = val
                if i < shingle_sz - 1:
                    continue

                point = tuple(shingle)
                for tree in forest:
                    # Remove oldest point if tree is full
                    if len(tree.leaves) > tree_size:
                        tree.forget_point(min(tree.leaves))
                    tree.insert_point(point, index=i)
                    if not tree.root:
                        continue
                    avg_codisp[i] += tree.codisp(i) / num_trees
        else:
            # ── Fallback: windowed isolation score ────────────────────────────
            from sklearn.ensemble import IsolationForest
            w = tree_size
            for i in range(n):
                if i < w:
                    continue
                window   = series[i - w: i].reshape(-1, 1)
                iso      = IsolationForest(n_estimators=20, contamination=0.05,
                                          random_state=42)
                iso.fit(window)
                score    = -iso.score_samples([[series[i]]])[0]
                avg_codisp[i] = score

        return avg_codisp


rrcf_detector = RRCFDetector(CFG)
df["det_rrcf"] = rrcf_detector.detect(df)
print(f"RRCF anomalies: {df['det_rrcf'].sum():,}  ({100*df['det_rrcf'].mean():.2f}%)")


12:32:13 | INFO     | detect.rrcf              | RRCF  trees=40  tree_size=256  shingle=8  pct=95
12:32:48 | INFO     | detect.rrcf              |   device=iPhone        threshold=47.456  flagged=150
12:33:21 | INFO     | detect.rrcf              |   device=MacBook       threshold=62.498  flagged=150
12:33:52 | INFO     | detect.rrcf              |   device=Apple Watch   threshold=50.218  flagged=150
12:35:42 | INFO     | detect.rrcf              |   device=iPad          threshold=67.143  flagged=150
12:35:42 | INFO     | detect.rrcf              |   total flagged 600 rows (5.00%)


RRCF anomalies: 600  (5.00%)


###  6 · Risk Scorer, CSAF Fusion & Ensemble Voting

In [21]:
# ── Risk Scorer ───────────────────────────────────────────────────────────────

class RiskScorer:
    """Normalised [0,1] multi-signal risk score using per-device percentile ranks."""

    def __init__(self, cfg: TelemetryConfig) -> None:
        self.cfg = cfg

    def score(self, df: pd.DataFrame) -> pd.Series:
        w = self.cfg.risk_weights
        risk = (
            w["latency"]     * df["latency_pct"]     +
            w["cpu_usage"]   * df["cpu_usage_pct"]   +
            w["temperature"] * df["temperature_pct"] +
            w["packet_loss"] * df["packet_loss_pct"] +
            w["battery_inv"] * (1 - df["battery_level"] / 100)
        ).clip(0, 1)
        # Boost by autoencoder reconstruction error
        if "ae_recon_error_z" in df.columns:
            boost = (df["ae_recon_error_z"].clip(0, 5) / 5) * 0.15
            risk  = (risk + boost).clip(0, 1)
        log.info("Risk  mean=%.3f  p95=%.3f  max=%.3f",
                 risk.mean(), risk.quantile(0.95), risk.max())
        return risk


# ── CSAF ──────────────────────────────────────────────────────────────────────

class CSAFStatus(str, Enum):
    CONFIRMED  = "CONFIRMED"
    PROBABLE   = "PROBABLE"
    SUPPRESSED = "SUPPRESSED"


class CSAFFusion:
    """Cross-Modal Semantic Anomaly Fusion — combines telemetry risk with UX signals."""

    def __init__(self, cfg: TelemetryConfig) -> None:
        self.cfg = cfg

    def enrich(self, df: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
        df = df.copy()
        df["review_score"]  = rng.normal(0.5, 0.2, len(df)).clip(0, 1)
        df["ticket_volume"] = rng.poisson(2, len(df)).astype(float)
        mask = df["anomaly_label"] == 1
        df.loc[mask, "review_score"]  += rng.uniform(0.2, 0.4, mask.sum())
        df.loc[mask, "ticket_volume"] += rng.integers(3, 12, mask.sum())
        df["review_score"]  = df["review_score"].clip(0, 1)
        df["review_norm"]   = df["review_score"] / df["review_score"].max()
        df["ticket_norm"]   = df["ticket_volume"] / df["ticket_volume"].max()
        w0, w1, w2 = self.cfg.csaf_weights
        df["csaf_score"] = (w0 * df["risk_score"] + w1 * df["review_norm"] + w2 * df["ticket_norm"]).clip(0, 1)
        df["csaf_anomaly"] = (df["csaf_score"] > self.cfg.csaf_threshold).astype(int)

        def _classify(row):
            hi_risk, hi_rev, hi_tix = row["risk_score"] > 0.70, row["review_norm"] > 0.60, row["ticket_norm"] > 0.60
            if hi_risk and hi_rev and hi_tix: return CSAFStatus.CONFIRMED
            if hi_risk and (hi_rev or hi_tix): return CSAFStatus.PROBABLE
            return CSAFStatus.SUPPRESSED

        df["csaf_status"] = df.apply(_classify, axis=1)
        log.info("CSAF counts: %s", df["csaf_status"].value_counts().to_dict())
        return df


# ── Ensemble ──────────────────────────────────────────────────────────────────

class EnsembleDetector:
    """Combines all 4 anomaly definitions + risk + CSAF via vote-weighted ensemble."""

    DETECTOR_COLS = [
        "det_sigma_data",   # σ from mean of data
        "det_sigma_errors", # σ from mean of errors
        "det_ndt",          # nonparametric dynamic thresholding
        "det_rrcf",         # RRCF scores
    ]

    def __init__(self, min_votes: int = 2) -> None:
        self.min_votes = min_votes

    def run(self, df: pd.DataFrame, cfg: TelemetryConfig) -> pd.DataFrame:
        log.info("Ensemble  min_votes=%d", self.min_votes)
        vote_cols = list(self.DETECTOR_COLS)

        df["vote_risk"] = (df["risk_score"] > cfg.risk_threshold).astype(int)
        df["vote_csaf"] = df["csaf_anomaly"]
        vote_cols += ["vote_risk", "vote_csaf"]

        df["vote_total"]    = df[vote_cols].sum(axis=1)
        df["vote_conf"]     = df["vote_total"] / len(vote_cols)
        df["final_anomaly"] = (df["vote_total"] >= self.min_votes).astype(int)

        df["severity"] = (
            0.50 * df["risk_score"] +
            0.30 * df["vote_conf"] +
            0.20 * df["csaf_score"]
        ).clip(0, 1)

        log.info("Ensemble → final_anomaly=%d (%.2f%%)  mean_severity=%.3f",
                 df["final_anomaly"].sum(), 100 * df["final_anomaly"].mean(),
                 df.loc[df["final_anomaly"]==1, "severity"].mean())
        return df


# ── Run ───────────────────────────────────────────────────────────────────────

scorer = RiskScorer(CFG)
df["risk_score"] = scorer.score(df)

csaf = CSAFFusion(CFG)
rng  = np.random.default_rng(CFG.random_seed + 99)
df   = csaf.enrich(df, rng)

ensemble = EnsembleDetector(min_votes=CFG.ensemble_min_votes)
df = ensemble.run(df, CFG)

vote_cols = [c for c in df.columns if c.startswith(("det_", "vote_")) and c not in ("vote_total", "vote_conf")]
print("\n── Vote Summary ──────────────────────────────")
for col in vote_cols:
    print(f"  {col:<22}  {int(df[col].sum()):5,}  ({100*df[col].mean():.2f}%)")
print(f"\n  Final anomalies flagged: {df['final_anomaly'].sum():,}")


12:36:19 | INFO     | telemetry.v3             | Risk  mean=0.487  p95=0.728  max=0.986
12:36:20 | INFO     | telemetry.v3             | CSAF counts: {<CSAFStatus.SUPPRESSED: 'SUPPRESSED'>: 11654, <CSAFStatus.PROBABLE: 'PROBABLE'>: 307, <CSAFStatus.CONFIRMED: 'CONFIRMED'>: 39}
12:36:20 | INFO     | telemetry.v3             | Ensemble  min_votes=2
12:36:20 | INFO     | telemetry.v3             | Ensemble → final_anomaly=558 (4.65%)  mean_severity=0.596



── Vote Summary ──────────────────────────────
  det_sigma_data            469  (3.91%)
  det_sigma_errors          352  (2.93%)
  det_ndt                   731  (6.09%)
  det_rrcf                  600  (5.00%)
  vote_risk                 900  (7.50%)
  vote_csaf                 312  (2.60%)

  Final anomalies flagged: 558


###  7 · Counts of Anomalous Points  *(Phase 2 output table)*

In [22]:
# ─────────────────────────────────────────────────────────────────────────────
# COUNTS OF ANOMALOUS POINTS — the key Phase-2 output table in the diagram.
# Rows  = detector / anomaly definition
# Cols  = device (and total)
# ─────────────────────────────────────────────────────────────────────────────

def build_anomaly_counts_table(df: pd.DataFrame) -> pd.DataFrame:
    detector_map = {
        "σ From Mean of Data"               : "det_sigma_data",
        "σ From Mean of Errors"             : "det_sigma_errors",
        "Nonparametric Dynamic Thresholding": "det_ndt",
        "RRCF Anomaly Score"                : "det_rrcf",
        "Risk Score Vote"                   : "vote_risk",
        "CSAF Vote"                         : "vote_csaf",
        "Ensemble Final"                    : "final_anomaly",
        "Ground Truth Injected"             : "anomaly_label",
    }

    rows = []
    for det_name, col in detector_map.items():
        if col not in df.columns:
            continue
        row = {"Detector / Definition": det_name}
        for device in sorted(df["device"].unique()):
            mask       = df["device"] == device
            row[device] = int(df.loc[mask, col].sum())
        row["TOTAL"] = int(df[col].sum())
        row["%"]     = f"{100 * df[col].mean():.2f}%"
        rows.append(row)

    counts_df = pd.DataFrame(rows)
    counts_df.to_csv(OUTPUT_DIR / "reports" / "anomaly_counts.csv", index=False)
    log.info("✓ Anomaly counts table saved")
    return counts_df


anomaly_counts = build_anomaly_counts_table(df)
print("\nCounts of Anomalous Points (Phase 2 output):")
print(anomaly_counts.to_string(index=False))


12:36:37 | INFO     | telemetry.v3             | ✓ Anomaly counts table saved



Counts of Anomalous Points (Phase 2 output):
             Detector / Definition  Apple Watch  MacBook  iPad  iPhone  TOTAL     %
               σ From Mean of Data          122      130   111     106    469 3.91%
             σ From Mean of Errors           89       88    92      83    352 2.93%
Nonparametric Dynamic Thresholding          191      172   184     184    731 6.09%
                RRCF Anomaly Score          150      150   150     150    600 5.00%
                   Risk Score Vote          245      249   199     207    900 7.50%
                         CSAF Vote           84       86    77      65    312 2.60%
                    Ensemble Final          145      146   134     133    558 4.65%
             Ground Truth Injected          150      150   150     150    600 5.00%


### 8 · Incident Clustering & Severity Tiering

In [23]:
class IncidentClusterer:
    """DBSCAN-based anomaly grouping with four severity tiers."""

    TIER_MAP = [("CRITICAL", 0.80), ("HIGH", 0.60), ("MEDIUM", 0.40), ("LOW", 0.00)]

    def __init__(self, cfg: TelemetryConfig) -> None:
        self.cfg = cfg

    def cluster(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df["incident_cluster"] = -2
        df["severity_tier"]    = "NORMAL"

        anomaly_mask = df["final_anomaly"] == 1
        adf = df.loc[anomaly_mask].copy()
        if len(adf) > self.cfg.dbscan_sample_size:
            adf = adf.sample(self.cfg.dbscan_sample_size, random_state=self.cfg.random_seed)

        feature_cols = ["latency", "cpu_usage", "temperature", "risk_score", "vote_conf"]
        feature_cols = [c for c in feature_cols if c in adf.columns]
        features     = adf[feature_cols].fillna(0)

        clustering = DBSCAN(eps=self.cfg.dbscan_eps, min_samples=self.cfg.dbscan_min_samples).fit(features)
        df.loc[adf.index, "incident_cluster"] = clustering.labels_

        for label in set(clustering.labels_):
            if label == -1: continue
            cidx      = adf.index[clustering.labels_ == label]
            mean_sev  = df.loc[cidx, "severity"].mean()
            tier      = next(t for t, thresh in self.TIER_MAP if mean_sev >= thresh)
            df.loc[cidx, "severity_tier"] = tier

        df.loc[df["incident_cluster"] == -1, "severity_tier"] = "LOW"
        n_clusters = len(set(clustering.labels_)) - (1 if -1 in clustering.labels_ else 0)
        log.info("Incidents: %d clusters + %d noise", n_clusters, (clustering.labels_ == -1).sum())
        return df


clusterer = IncidentClusterer(CFG)
df = clusterer.cluster(df)
print("Severity tier distribution:")
print(df["severity_tier"].value_counts().to_string())


12:36:47 | INFO     | telemetry.v3             | Incidents: 1 clusters + 154 noise


Severity tier distribution:
severity_tier
NORMAL    11442
MEDIUM      404
LOW         154


### 9 · Model Evaluation

In [24]:
class PipelineEvaluator:
    """Precision / Recall / F1 / ROC-AUC evaluation for each detector and the ensemble."""

    def __init__(self, cfg: TelemetryConfig) -> None:
        self.cfg = cfg

    def evaluate(self, df: pd.DataFrame) -> Dict:
        y_true  = df["anomaly_label"]
        y_pred  = df["final_anomaly"]
        y_score = df["risk_score"]

        roc_auc = roc_auc_score(y_true, y_score)
        avg_pr  = average_precision_score(y_true, y_score)
        report  = classification_report(y_true, y_pred, output_dict=True)
        cm      = confusion_matrix(y_true, y_pred)
        tn, fp, fn, tp = cm.ravel()
        fpr = fp / (fp + tn) if (fp + tn) else 0
        fnr = fn / (fn + tp) if (fn + tp) else 0

        metrics = {
            "roc_auc": roc_auc, "avg_precision": avg_pr,
            "precision": report["1"]["precision"], "recall": report["1"]["recall"],
            "f1": report["1"]["f1-score"],
            "false_positive_rate": fpr, "false_negative_rate": fnr,
            "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
        }

        sep = "─" * 50
        print(f"\n{sep}\n  PIPELINE EVALUATION REPORT\n{sep}")
        print(f"  ROC-AUC              : {roc_auc:.4f}")
        print(f"  Average Precision    : {avg_pr:.4f}")
        print(f"  Precision            : {metrics['precision']:.4f}")
        print(f"  Recall               : {metrics['recall']:.4f}")
        print(f"  F1-Score             : {metrics['f1']:.4f}")
        print(f"  False Positive Rate  : {fpr:.4f}")
        print(f"  False Negative Rate  : {fnr:.4f}")
        print(f"{sep}\n  Confusion Matrix")
        print(f"    TP={tp:6,}   FP={fp:6,}")
        print(f"    FN={fn:6,}   TN={tn:6,}")
        print(f"\n  Per-Detector Metrics:")
        det_cols = {
            "σ-Data"   : "det_sigma_data",
            "σ-Errors" : "det_sigma_errors",
            "NDT"      : "det_ndt",
            "RRCF"     : "det_rrcf",
            "Risk-Vote": "vote_risk",
            "CSAF-Vote": "vote_csaf",
        }
        for name, col in det_cols.items():
            if col not in df.columns: continue
            rep = classification_report(y_true, df[col], output_dict=True, zero_division=0)
            p   = rep.get("1", {}).get("precision", 0)
            r   = rep.get("1", {}).get("recall", 0)
            f1  = rep.get("1", {}).get("f1-score", 0)
            print(f"    {name:<12}  P={p:.3f}  R={r:.3f}  F1={f1:.3f}")
        print(sep)
        return metrics


evaluator = PipelineEvaluator(CFG)
metrics   = evaluator.evaluate(df)



──────────────────────────────────────────────────
  PIPELINE EVALUATION REPORT
──────────────────────────────────────────────────
  ROC-AUC              : 0.7123
  Average Precision    : 0.1686
  Precision            : 0.6326
  Recall               : 0.5883
  F1-Score             : 0.6097
  False Positive Rate  : 0.0180
  False Negative Rate  : 0.4117
──────────────────────────────────────────────────
  Confusion Matrix
    TP=   353   FP=   205
    FN=   247   TN=11,195

  Per-Detector Metrics:
    σ-Data        P=0.814  R=0.637  F1=0.715
    σ-Errors      P=0.798  R=0.468  F1=0.590
    NDT           P=0.146  R=0.178  F1=0.161
    RRCF          P=0.172  R=0.172  F1=0.172
    Risk-Vote     P=0.171  R=0.257  F1=0.205
    CSAF-Vote     P=0.833  R=0.433  F1=0.570
──────────────────────────────────────────────────


###  10 · Interactive Visualizations

In [25]:
PALETTE = {
    "bg": "#060d14", "paper": "#0b1520", "grid": "#0e2235",
    "text": "#8aaccc", "cyan": "#00e5ff", "amber": "#ffb300",
    "red": "#ff2d55", "green": "#10b981", "purple": "#7c3aed",
    "orange": "#f97316",
}
_LAYOUT = dict(
    paper_bgcolor=PALETTE["paper"], plot_bgcolor=PALETTE["bg"],
    font=dict(family="Share Tech Mono, monospace", color=PALETTE["text"], size=11),
    xaxis=dict(gridcolor=PALETTE["grid"]), yaxis=dict(gridcolor=PALETTE["grid"]),
    margin=dict(l=60, r=30, t=55, b=50),
    legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(color=PALETTE["text"])),
)
def _layout(fig, title, **kw):
    fig.update_layout(title=dict(text=title, font=dict(color=PALETTE["cyan"], size=14)), **_LAYOUT, **kw)
    return fig
print("Palette & layout helpers ready — running plots ...")


Palette & layout helpers ready — running plots ...


In [26]:
# ── Plot 1: Multi-Model Fit — Latency Timeline ───────────────────────────────
dev  = "iPhone"
sub  = df[df["device"] == dev].iloc[:1500].copy()

fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=sub["timestamp"], y=sub["latency"],
    mode="lines", name="Raw Latency",
    line=dict(color=PALETTE["cyan"], width=1), opacity=0.45))
fig1.add_trace(go.Scatter(x=sub["timestamp"], y=sub["rm_pred_latency"],
    mode="lines", name="Rolling Mean",
    line=dict(color=PALETTE["amber"], width=2)))
if "arima_pred_latency" in sub.columns:
    fig1.add_trace(go.Scatter(x=sub["timestamp"], y=sub["arima_pred_latency"],
        mode="lines", name="ARIMA",
        line=dict(color=PALETTE["green"], width=2, dash="dot")))
# Anomaly overlays
det = sub[sub["final_anomaly"] == 1]
fig1.add_trace(go.Scatter(x=det["timestamp"], y=det["latency"],
    mode="markers", name="Ensemble Anomaly",
    marker=dict(color=PALETTE["red"], size=7, symbol="x")))
tp = sub[(sub["final_anomaly"]==1) & (sub["anomaly_label"]==1)]
fig1.add_trace(go.Scatter(x=tp["timestamp"], y=tp["latency"],
    mode="markers", name="True Positive",
    marker=dict(color=PALETTE["green"], size=9, symbol="circle-open", line=dict(width=2))))
fig1 = _layout(fig1, "Phase 1 Models of Normal — Latency Fit (iPhone)")
fig1.show()


In [27]:
# ── Plot 2: ARIMA Residuals vs Rolling Residuals ─────────────────────────────
fig2 = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=("Rolling Mean Residuals", "ARIMA Residuals"),
    vertical_spacing=0.12)

fig2.add_trace(go.Scatter(x=sub["timestamp"], y=sub["rm_resid_latency"],
    mode="lines", name="RM Residual", line=dict(color=PALETTE["amber"], width=1)), row=1, col=1)
if "arima_resid_latency" in sub.columns:
    fig2.add_trace(go.Scatter(x=sub["timestamp"], y=sub["arima_resid_latency"],
        mode="lines", name="ARIMA Residual", line=dict(color=PALETTE["purple"], width=1)), row=2, col=1)

# Highlight anomalies on residual plot
for row_i, col_name in [(1, "rm_resid_latency"), (2, "arima_resid_latency")]:
    if col_name in sub.columns:
        anom = sub[sub["anomaly_label"] == 1]
        fig2.add_trace(go.Scatter(x=anom["timestamp"], y=anom[col_name],
            mode="markers", name=f"True Anom ({col_name[:2]})",
            marker=dict(color=PALETTE["red"], size=6, symbol="x")), row=row_i, col=1)

fig2.update_layout(paper_bgcolor=PALETTE["paper"], plot_bgcolor=PALETTE["bg"],
    font=dict(family="Share Tech Mono", color=PALETTE["text"]),
    title=dict(text="Phase 1 — Model Residuals (Rolling Mean vs ARIMA)",
               font=dict(color=PALETTE["cyan"], size=14)),
    height=500, showlegend=True, margin=dict(l=60, r=30, t=70, b=50))
fig2.update_xaxes(gridcolor=PALETTE["grid"])
fig2.update_yaxes(gridcolor=PALETTE["grid"])
fig2.show()


In [28]:
# ── Plot 3: Autoencoder — Reconstruction Error + Latent Space ────────────────
ae_sub = df[df["device"] == dev].copy()
fig3   = make_subplots(rows=1, cols=2,
    subplot_titles=("Reconstruction Error vs Anomaly Label",
                    "Latent Space (dim-0 vs dim-1, coloured by anomaly)"))

# Left: error time series
ae_sub["ae_recon_smooth"] = ae_sub["ae_recon_error"].rolling(30).mean()
fig3.add_trace(go.Scatter(
    x=ae_sub["timestamp"].iloc[:2000], y=ae_sub["ae_recon_error"].iloc[:2000],
    mode="lines", name="Recon Error", line=dict(color=PALETTE["purple"], width=1), opacity=0.5
), row=1, col=1)
fig3.add_trace(go.Scatter(
    x=ae_sub["timestamp"].iloc[:2000], y=ae_sub["ae_recon_smooth"].iloc[:2000],
    mode="lines", name="Smoothed", line=dict(color=PALETTE["amber"], width=2)
), row=1, col=1)

# Right: latent scatter
if "ae_latent_0" in ae_sub.columns and "ae_latent_1" in ae_sub.columns:
    samp = ae_sub.sample(min(1000, len(ae_sub)), random_state=42)
    colors = [PALETTE["red"] if a else PALETTE["cyan"] for a in samp["anomaly_label"]]
    fig3.add_trace(go.Scatter(
        x=samp["ae_latent_0"], y=samp["ae_latent_1"],
        mode="markers", name="Latent Points",
        marker=dict(color=colors, size=4, opacity=0.7)
    ), row=1, col=2)

fig3.update_layout(paper_bgcolor=PALETTE["paper"], plot_bgcolor=PALETTE["bg"],
    font=dict(family="Share Tech Mono", color=PALETTE["text"]),
    title=dict(text="Autoencoder — Compressed Feature Vectors & Reconstruction Error",
               font=dict(color=PALETTE["cyan"], size=14)),
    height=420, margin=dict(l=60, r=30, t=70, b=50))
fig3.update_xaxes(gridcolor=PALETTE["grid"])
fig3.update_yaxes(gridcolor=PALETTE["grid"])
fig3.show()


In [29]:
# ── Plot 4: Anomaly Definition Comparison Heatmap ─────────────────────────────
det_cols_map = {
    "σ-Data"   : "det_sigma_data",
    "σ-Errors" : "det_sigma_errors",
    "NDT"      : "det_ndt",
    "RRCF"     : "det_rrcf",
    "Risk-Vote": "vote_risk",
    "CSAF-Vote": "vote_csaf",
    "Ensemble" : "final_anomaly",
    "Truth"    : "anomaly_label",
}
cols_present = {k: v for k, v in det_cols_map.items() if v in df.columns}
det_matrix   = df[[v for v in cols_present.values()]].rename(columns={v: k for k, v in cols_present.items()})
corr_det     = det_matrix.corr()

fig4 = go.Figure(go.Heatmap(
    z=corr_det.values, x=corr_det.columns.tolist(), y=corr_det.index.tolist(),
    colorscale=[[0, PALETTE["bg"]], [0.5, PALETTE["purple"]], [1, PALETTE["cyan"]]],
    text=corr_det.round(2).astype(str).values, texttemplate="%{text}", showscale=True,
    zmin=-0.1, zmax=1.0,
))
fig4 = _layout(fig4, "Anomaly Definition Agreement Matrix (Pearson r)")
fig4.show()


In [30]:
# ── Plot 5: Counts of Anomalous Points by Detector & Device ─────────────────
count_plot_data = []
for det_name, col in det_cols_map.items():
    if col not in df.columns: continue
    for device in sorted(df["device"].unique()):
        mask = df["device"] == device
        count_plot_data.append({
            "Detector": det_name, "Device": device, "Count": int(df.loc[mask, col].sum())
        })
count_plot_df = pd.DataFrame(count_plot_data)

fig5 = px.bar(count_plot_df, x="Detector", y="Count", color="Device", barmode="group",
    color_discrete_sequence=[PALETTE["cyan"], PALETTE["amber"], PALETTE["green"], PALETTE["purple"]])
fig5 = _layout(fig5, "Phase 2 Output — Counts of Anomalous Points by Detector & Device")
fig5.show()


In [31]:
# ── Plot 6: ROC + Precision-Recall Curves ─────────────────────────────────────
fpr_arr, tpr_arr, _ = roc_curve(df["anomaly_label"], df["risk_score"])
prec, rec, _        = precision_recall_curve(df["anomaly_label"], df["risk_score"])
roc_val, ap_val     = metrics["roc_auc"], metrics["avg_precision"]

fig6 = make_subplots(rows=1, cols=2,
    subplot_titles=(f"ROC Curve (AUC={roc_val:.3f})", f"Precision-Recall (AP={ap_val:.3f})"))
fig6.add_trace(go.Scatter(x=fpr_arr, y=tpr_arr, mode="lines", name="ROC",
    line=dict(color=PALETTE["cyan"], width=2)), row=1, col=1)
fig6.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Chance",
    line=dict(color=PALETTE["text"], dash="dash", width=1)), row=1, col=1)
fig6.add_trace(go.Scatter(x=rec, y=prec, mode="lines", name="PR Curve",
    line=dict(color=PALETTE["amber"], width=2)), row=1, col=2)
fig6.update_layout(paper_bgcolor=PALETTE["paper"], plot_bgcolor=PALETTE["bg"],
    font=dict(family="Share Tech Mono", color=PALETTE["text"]),
    title=dict(text="Model Evaluation Curves", font=dict(color=PALETTE["cyan"], size=14)),
    margin=dict(l=60, r=30, t=70, b=50))
fig6.update_xaxes(gridcolor=PALETTE["grid"])
fig6.update_yaxes(gridcolor=PALETTE["grid"])
fig6.show()


In [32]:
# ── Plot 7: Severity Tier Distribution ───────────────────────────────────────
tier_colors = {"CRITICAL": PALETTE["red"], "HIGH": PALETTE["amber"],
               "MEDIUM": PALETTE["purple"], "LOW": PALETTE["green"], "NORMAL": PALETTE["text"]}
sev_counts = df["severity_tier"].value_counts().reset_index()
sev_counts.columns = ["tier", "count"]

fig7 = go.Figure(go.Bar(
    x=sev_counts["tier"], y=sev_counts["count"],
    marker_color=[tier_colors.get(t, PALETTE["text"]) for t in sev_counts["tier"]],
    text=sev_counts["count"], textposition="outside", textfont=dict(color=PALETTE["text"])))
fig7 = _layout(fig7, "Incident Severity Tier Distribution")
fig7.show()


In [33]:
# ── Plot 8: Correlation Coefficients (Phase 1 output) ─────────────────────────
sig_list = ["latency", "cpu_usage", "temperature"]
model_list = ["Rolling Mean", "ARIMA"]
corr_hm_data = []
for dev_name in df["device"].unique():
    mask = df["device"] == dev_name
    sub  = df.loc[mask]
    for sig in sig_list:
        for mdl in model_list:
            pred_col = f"rm_pred_{sig}" if mdl == "Rolling Mean" else f"arima_pred_{sig}"
            if pred_col in sub.columns:
                r = round(sub[sig].corr(sub[pred_col]), 4)
                corr_hm_data.append({"Device_Model": f"{dev_name}\n{mdl}", "Signal": sig, "r": r})

corr_hm = pd.DataFrame(corr_hm_data)
fig8 = px.density_heatmap(corr_hm, x="Signal", y="Device_Model", z="r",
    color_continuous_scale=[[0, PALETTE["bg"]], [0.5, PALETTE["purple"]], [1, PALETTE["cyan"]]],
    text_auto=".3f")
fig8.update_layout(paper_bgcolor=PALETTE["paper"], plot_bgcolor=PALETTE["bg"],
    font=dict(family="Share Tech Mono", color=PALETTE["text"]),
    title=dict(text="Phase 1 Output — Correlation Coefficients (Model vs Raw Signal)",
               font=dict(color=PALETTE["cyan"], size=14)),
    margin=dict(l=80, r=30, t=70, b=50))
fig8.show()


In [34]:
# ── Plot 9: RRCF CoDisp Score Timeline ──────────────────────────────────────
if "rrcf_codisp" in df.columns:
    rrcf_sub = df[df["device"] == dev].iloc[:2000].copy()
    threshold_val = np.percentile(rrcf_sub["rrcf_codisp"], CFG.rrcf_threshold_pct)

    fig9 = go.Figure()
    fig9.add_trace(go.Scatter(x=rrcf_sub["timestamp"], y=rrcf_sub["rrcf_codisp"],
        mode="lines", name="CoDisp Score", line=dict(color=PALETTE["amber"], width=1.5)))
    fig9.add_hrect(y0=threshold_val, y1=rrcf_sub["rrcf_codisp"].max() * 1.05,
        fillcolor=PALETTE["red"], opacity=0.08, line_width=0,
        annotation_text=f"p{CFG.rrcf_threshold_pct:.0f} threshold",
        annotation_font_color=PALETTE["red"])
    rrcf_anom = rrcf_sub[rrcf_sub["det_rrcf"] == 1]
    fig9.add_trace(go.Scatter(x=rrcf_anom["timestamp"], y=rrcf_anom["rrcf_codisp"],
        mode="markers", name="RRCF Anomaly",
        marker=dict(color=PALETTE["red"], size=7, symbol="x")))
    fig9 = _layout(fig9, "RRCF CoDisp Anomaly Scores — iPhone")
    fig9.show()


###  11 · Alert Manager

In [35]:
@dataclass
class Alert:
    alert_id     : str
    timestamp    : pd.Timestamp
    device       : str
    tier         : str
    severity     : float
    anomaly_type : str
    csaf_status  : str
    vote_total   : int
    risk_score   : float
    detectors    : str   # which definitions fired

    def summary(self) -> str:
        return (
            f"[{self.tier:8s}] {self.timestamp.strftime('%H:%M:%S')}  "
            f"device={self.device:<12s}  type={self.anomaly_type:<18s}  "
            f"sev={self.severity:.3f}  votes={self.vote_total}  "
            f"det=[{self.detectors}]"
        )


class AlertManager:
    """Generates priority-tiered, ranked alerts from ensemble output."""

    TIER_ORDER = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3}
    DET_MAP    = {
        "det_sigma_data":   "σ-Data",
        "det_sigma_errors": "σ-Err",
        "det_ndt":          "NDT",
        "det_rrcf":         "RRCF",
    }

    def __init__(self, cfg: TelemetryConfig) -> None:
        self.cfg    = cfg
        self.alerts : List[Alert] = []

    def generate(self, df: pd.DataFrame) -> List[Alert]:
        self.alerts.clear()
        flagged = df[df["final_anomaly"] == 1].copy()

        for i, (_, row) in enumerate(flagged.iterrows()):
            fired = [short for col, short in self.DET_MAP.items()
                     if col in row.index and row[col] == 1]
            alert = Alert(
                alert_id     = f"ALT-{i:05d}",
                timestamp    = row["timestamp"],
                device       = row["device"],
                tier         = row["severity_tier"],
                severity     = row["severity"],
                anomaly_type = row["anomaly_type"],
                csaf_status  = row["csaf_status"],
                vote_total   = int(row["vote_total"]),
                risk_score   = row["risk_score"],
                detectors    = ",".join(fired) if fired else "ensemble",
            )
            self.alerts.append(alert)

        self.alerts.sort(key=lambda a: (self.TIER_ORDER.get(a.tier, 99), -a.severity))
        # Save
        alert_dicts = [vars(a) for a in self.alerts]
        pd.DataFrame(alert_dicts).to_csv(OUTPUT_DIR / "anomalies" / "alerts.csv", index=False)
        log.info("Alerts: %d  saved → alerts.csv", len(self.alerts))
        return self.alerts

    def top_n(self, n: int = 20) -> List[Alert]: return self.alerts[:n]
    def by_tier(self) -> Dict[str, int]:
        counts: Dict[str, int] = {}
        for a in self.alerts:
            counts[a.tier] = counts.get(a.tier, 0) + 1
        return counts


am = AlertManager(CFG)
am.generate(df)

print("\nAlert Summary:")
for tier, cnt in am.by_tier().items():
    bar = "█" * min(cnt // 10, 50)
    print(f"  {tier:8s}  {cnt:5,}  {bar}")
print(f"\nTop 15 Alerts:")
print("─" * 110)
for a in am.top_n(15):
    print(" ", a.summary())
print("─" * 110)


12:38:23 | INFO     | telemetry.v3             | Alerts: 558  saved → alerts.csv



Alert Summary:
  MEDIUM      404  ████████████████████████████████████████
  LOW         154  ███████████████

Top 15 Alerts:
──────────────────────────────────────────────────────────────────────────────────────────────────────────────
  [MEDIUM  ] 00:32:09  device=iPad          type=overheat            sev=0.861  votes=5  det=[σ-Data,σ-Err,NDT]
  [MEDIUM  ] 00:06:43  device=iPad          type=cpu_spike           sev=0.859  votes=6  det=[σ-Data,σ-Err,NDT,RRCF]
  [MEDIUM  ] 00:06:14  device=iPhone        type=cpu_spike           sev=0.835  votes=5  det=[σ-Data,σ-Err,NDT]
  [MEDIUM  ] 00:32:52  device=Apple Watch   type=cpu_spike           sev=0.825  votes=5  det=[σ-Data,σ-Err,NDT]
  [MEDIUM  ] 00:32:29  device=iPhone        type=cpu_spike           sev=0.818  votes=4  det=[σ-Data,σ-Err]
  [MEDIUM  ] 00:12:57  device=Apple Watch   type=network_issue       sev=0.806  votes=3  det=[σ-Data]
  [MEDIUM  ] 00:17:43  device=iPad          type=cpu_spike           sev=0.797  votes=5  det=[σ-Dat

### 12 · Pipeline Summary Report

In [38]:
report_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
sep  = "═" * 65
sep2 = "─" * 65

report_lines = [
    f"",
    f"{sep}",
    f"  TELEMETRY ANOMALY DETECTION  v3.0  —  PIPELINE REPORT",
    f"  Generated : {report_time}",
    f"{sep}",
    f"",
    f"  ARCHITECTURE",
    f"{sep2}",
    f"  Phase 1 — Models of Normal",
    f"     Rolling Mean Model       (window={CFG.rolling_window})",
    f"     ARIMA{CFG.arima_order} Model",
    f"     Autoencoder              (latent={CFG.ae_latent_dim}-D compressed feature vectors)",
    f"  Phase 2 — Anomaly Definitions",
    f"     σ's From Mean of Data    (k={CFG.sigma_data_k})",
    f"     σ's From Mean of Errors  (k={CFG.sigma_error_k})",
    f"     Nonparametric Dynamic Thresholding  (window={CFG.ndt_window})",
    f"     RRCF Anomaly Scores      (trees={CFG.rrcf_num_trees}, shingle={CFG.rrcf_shingle_size})",
    f"  Fusion",
    f"     Risk Scorer + CSAF + Ensemble Voting (min_votes={CFG.ensemble_min_votes})",
    f"",
    f"  DATASET",
    f"{sep2}",
    f"  Total rows          : {len(df):,}",
    f"  Devices             : {', '.join(CFG.devices)}",
    f"  OS versions         : {', '.join(CFG.os_versions)}",
    f"  Injected anomalies  : {df['anomaly_label'].sum():,}  ({100*df['anomaly_label'].mean():.1f}%)",
    f"",
    f"  PHASE 1 — CORRELATION COEFFICIENTS",
    f"{sep2}",
]

for _, row in corr_df.groupby(["model", "signal"])["pearson_r"].mean().reset_index().iterrows():
    report_lines.append(f"  {row['model']:<20} {row['signal']:<14} r={row['pearson_r']:.4f}")

report_lines += [
    f"",
    f"  PHASE 2 — ANOMALY DETECTION RESULTS",
    f"{sep2}",
    f"  Final anomalies     : {df['final_anomaly'].sum():,}  ({100*df['final_anomaly'].mean():.1f}%)",
    f"  ROC-AUC             : {metrics['roc_auc']:.4f}",
    f"  Average Precision   : {metrics['avg_precision']:.4f}",
    f"  F1-Score            : {metrics['f1']:.4f}",
    f"  Precision           : {metrics['precision']:.4f}",
    f"  Recall              : {metrics['recall']:.4f}",
    f"  False Positive Rate : {metrics['false_positive_rate']:.4f}",
    f"",
    f"  COUNTS OF ANOMALOUS POINTS",
    f"{sep2}",
]
for _, row in anomaly_counts.iterrows():
    report_lines.append(f"  {row['Detector / Definition']:<38} Total={row['TOTAL']:>6,}  ({row['%']})")

report_lines += [
    f"",
    f"  SEVERITY BREAKDOWN",
    f"{sep2}",
]
for tier in ["CRITICAL", "HIGH", "MEDIUM", "LOW", "NORMAL"]:
    cnt = (df["severity_tier"] == tier).sum()
    pct = 100 * cnt / len(df)
    report_lines.append(f"  {tier:8s}  {cnt:7,}  ({pct:5.2f}%)")

report_lines += [
    f"",
    f"  ALERTS",
    f"{sep2}",
    f"  Total alerts        : {len(am.alerts):,}",
]
for tier, cnt in am.by_tier().items():
    report_lines.append(f"  {tier:8s}  {cnt:7,}")

report_lines += [
    f"",
    f"  OUTPUTS SAVED TO → {OUTPUT_DIR}/",
    f"{sep2}",
    f"  models/rolling_mean_stats.csv",
    f"  models/arima_fits.csv",
    f"  models/arima_models.pkl",
    f"  models/autoencoder_models.pkl",
    f"  features/compressed_feature_vectors.csv",
    f"  reports/correlation_coefficients.csv",
    f"  reports/anomaly_counts.csv",
    f"  anomalies/alerts.csv",
    f"",
    f"{sep}",
    f"  Pipeline complete.",
    f"{sep}",
    f"",
]

report_text = "\n".join(report_lines)
print(report_text)

# Save report
with open(OUTPUT_DIR / "reports" / "pipeline_report.txt", "w", encoding="utf-8") as f:
    f.write(report_text)
log.info(" Summary report saved → pipeline_report.txt")


12:39:55 | INFO     | telemetry.v3             |  Summary report saved → pipeline_report.txt



═════════════════════════════════════════════════════════════════
  TELEMETRY ANOMALY DETECTION  v3.0  —  PIPELINE REPORT
  Generated : 2026-04-29 12:39:55
═════════════════════════════════════════════════════════════════

  ARCHITECTURE
─────────────────────────────────────────────────────────────────
  Phase 1 — Models of Normal
     Rolling Mean Model       (window=50)
     ARIMA(2, 1, 2) Model
     Autoencoder              (latent=8-D compressed feature vectors)
  Phase 2 — Anomaly Definitions
     σ's From Mean of Data    (k=3.0)
     σ's From Mean of Errors  (k=3.0)
     Nonparametric Dynamic Thresholding  (window=50)
     RRCF Anomaly Scores      (trees=40, shingle=8)
  Fusion
     Risk Scorer + CSAF + Ensemble Voting (min_votes=2)

  DATASET
─────────────────────────────────────────────────────────────────
  Total rows          : 12,000
  Devices             : iPhone, MacBook, Apple Watch, iPad
  OS versions         : iOS 18.3, iOS 18.4, iOS 18.5
  Injected anomalies  : 600  (